# Romberg Test: Eyes Open vs Eyes Closed Classifier

Train a logistic regression model to distinguish **eyes open** vs **eyes closed** standing balance from raw smartphone accelerometer data. This is the digital Romberg test.

In [ ]:
# Cell 1: Imports & Configuration
import pandas as pd
import numpy as np
import json
import os
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

# Constants
DATA_DIR = 'balance_data'
SAMPLE_RATE = 100      # ~100 Hz from Sensor Logger
WINDOW_SIZE = 3000     # 30 seconds at 100 Hz
WINDOW_STRIDE = 1500   # 15 seconds (50% overlap)
MIN_WINDOW = 1500      # minimum 15 seconds
TRIM_SEC = 5           # trim first/last 5 seconds (settling time)
TRIM_SAMPLES = TRIM_SEC * SAMPLE_RATE

print('Configuration ready.')
print(f'  Window: {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLE_RATE}s), stride: {WINDOW_STRIDE} ({WINDOW_STRIDE/SAMPLE_RATE}s)')
print(f'  Trimming first/last {TRIM_SEC}s from each recording')

In [ ]:
# Cell 2: Load Data
# Filenames follow pattern: Accelerometer_<SubjectName>_<Open|Closed>.csv
raw_data = []

for label in ['eyes_open', 'eyes_closed']:
    folder = os.path.join(DATA_DIR, label)
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(folder, fname)
        df = pd.read_csv(fpath)
        
        # Extract subject name from filename
        match = re.search(r'Accelerometer_(.+?)_(Open|Closed)', fname, re.IGNORECASE)
        subject = match.group(1) if match else fname.replace('.csv', '')
        
        # Estimate actual sample rate
        if 'seconds_elapsed' in df.columns:
            dt = df['seconds_elapsed'].diff().median()
            actual_rate = round(1.0 / dt)
            duration = df['seconds_elapsed'].iloc[-1]
            print(f'  {label}/{fname}: subject={subject}, {len(df)} rows, ~{actual_rate} Hz, {duration:.1f}s')
        
        raw_data.append({
            'filename': fname,
            'subject': subject,
            'label': label,
            'df': df,
        })

subjects = sorted(set(d['subject'] for d in raw_data))
print(f'\nLoaded {len(raw_data)} files from {len(subjects)} subjects: {subjects}')
for label in ['eyes_open', 'eyes_closed']:
    print(f'  {label}: {sum(1 for d in raw_data if d["label"]==label)}')

In [ ]:
# Cell 3: Feature Extraction
# Raw accelerometer includes gravity. We subtract the per-window mean
# from each axis to isolate the dynamic (sway) component.

def extract_features(ax, ay, az, sample_rate=100):
    """Extract 8 features from acceleration arrays."""
    n = len(ax)
    if n < 10:
        return None
    
    # Remove gravity by subtracting per-window mean
    ax = ax - np.mean(ax)
    ay = ay - np.mean(ay)
    az = az - np.mean(az)
    
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    
    dx = np.diff(ax)
    dy = np.diff(ay)
    dz = np.diff(az)
    jerk = np.sqrt(dx**2 + dy**2 + dz**2) * sample_rate
    
    displacements = np.sqrt(dx**2 + dy**2 + dz**2)
    path_length = np.sum(displacements) / n
    
    return {
        'rms_sway': np.sqrt(np.mean(mag**2)),
        'std_x': np.std(ax, ddof=0),
        'std_y': np.std(ay, ddof=0),
        'std_z': np.std(az, ddof=0),
        'mean_jerk': np.mean(jerk),
        'path_length': path_length,
        'sway_mean': np.mean(mag),
        'sway_peak': np.max(mag),
    }

def segment_and_extract(df, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE,
                         min_window=MIN_WINDOW, sample_rate=SAMPLE_RATE,
                         trim=TRIM_SAMPLES):
    """Window a dataframe and extract features from each window."""
    ax = df['x'].values
    ay = df['y'].values
    az = df['z'].values
    
    # Trim first/last 5 seconds
    if len(ax) > 2 * trim + min_window:
        ax = ax[trim:-trim]
        ay = ay[trim:-trim]
        az = az[trim:-trim]
    
    n = len(ax)
    windows = []
    start = 0
    while start + min_window <= n:
        end = min(start + window_size, n)
        feats = extract_features(ax[start:end], ay[start:end], az[start:end], sample_rate)
        if feats is not None:
            feats['window_start'] = start
            feats['window_samples'] = end - start
            windows.append(feats)
        start += stride
    return windows

# Build feature matrix
rows = []
for entry in raw_data:
    windows = segment_and_extract(entry['df'])
    for w in windows:
        w['filename'] = entry['filename']
        w['subject'] = entry['subject']
        w['label'] = entry['label']
        rows.append(w)

feature_df = pd.DataFrame(rows)
FEATURE_NAMES = ['rms_sway', 'std_x', 'std_y', 'std_z', 'mean_jerk', 'path_length', 'sway_mean', 'sway_peak']

print(f'Feature matrix: {feature_df.shape}')
print(f'\nClass distribution:')
print(feature_df['label'].value_counts())
print(f'\nWindows per subject:')
print(feature_df.groupby('subject').size())
print(f'\nWindows per subject x label:')
print(feature_df.groupby(['subject', 'label']).size().unstack(fill_value=0))

In [ ]:
# Cell 4: EDA — Feature distributions by class
colors = {'eyes_open': '#2980b9', 'eyes_closed': '#c0392b'}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flat, FEATURE_NAMES):
    sns.boxplot(data=feature_df, x='label', y=feat, ax=ax, palette=colors, width=0.5)
    ax.set_title(feat.replace('_', ' '), fontsize=10)
    ax.set_xlabel('')

plt.tight_layout()
plt.suptitle('Feature Distributions: Eyes Open vs Closed', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Cell 5: Cross-Validation
# With 2+ subjects: use Leave-One-Subject-Out (gold standard)
# With 1 subject: fall back to stratified k-fold

X = feature_df[FEATURE_NAMES].values
y = feature_df['label'].values
groups = feature_df['subject'].values

n_subjects = len(set(groups))
y_pred_all = np.empty_like(y, dtype=object)
per_subject_results = []

if n_subjects >= 2:
    print(f'Using Leave-One-Subject-Out CV ({n_subjects} subjects)')
    logo = LeaveOneGroupOut()
    splitter = logo.split(X, y, groups)
else:
    print('Only 1 subject — falling back to 5-fold stratified CV')
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splitter = skf.split(X, y)

for train_idx, test_idx in splitter:
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_train_s, y_train)
    
    y_pred = clf.predict(X_test_s)
    y_pred_all[test_idx] = y_pred
    
    if n_subjects >= 2:
        subj = groups[test_idx[0]]
        acc = accuracy_score(y_test, y_pred)
        per_subject_results.append({'subject': subj, 'accuracy': acc, 'n_samples': len(y_test)})

overall_acc = accuracy_score(y, y_pred_all)
cv_type = 'Leave-One-Subject-Out' if n_subjects >= 2 else '5-Fold Stratified'
print(f'\n=== {cv_type} Cross-Validation ===')
print(f'Overall Accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)\n')
print(classification_report(y, y_pred_all, target_names=['eyes_closed', 'eyes_open']))

if per_subject_results:
    print('\nPer-subject accuracy:')
    for r in sorted(per_subject_results, key=lambda x: x['subject']):
        print(f'  {r["subject"]}: {r["accuracy"]*100:.1f}% ({r["n_samples"]} windows)')

In [ ]:
# Cell 6: Confusion Matrix & Per-Subject Accuracy
n_plots = 2 if per_subject_results else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

cm = confusion_matrix(y, y_pred_all, labels=['eyes_closed', 'eyes_open'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Eyes Closed', 'Eyes Open'],
            yticklabels=['Eyes Closed', 'Eyes Open'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix ({cv_type}) — {overall_acc*100:.1f}%')

if per_subject_results:
    subj_df = pd.DataFrame(per_subject_results).sort_values('subject')
    bar_colors = ['#2d5a3d' if a >= 0.7 else '#e67e22' if a >= 0.5 else '#c0392b'
                  for a in subj_df['accuracy']]
    axes[1].bar(subj_df['subject'], subj_df['accuracy'] * 100, color=bar_colors)
    axes[1].set_xlabel('Subject')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Per-Subject Classification Accuracy')
    axes[1].axhline(y=overall_acc * 100, color='gray', linestyle='--', linewidth=1,
                    label=f'Mean: {overall_acc*100:.1f}%')
    axes[1].legend()
    axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Train final model on ALL data & export weights
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X)

final_model = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
final_model.fit(X_scaled, y)

print(f'Final model classes: {list(final_model.classes_)}')
print(f'Final model intercept: {final_model.intercept_[0]:.10f}')
print(f'\nFeature weights:')
for name, weight in zip(FEATURE_NAMES, final_model.coef_[0]):
    print(f'  {name:15s}: {weight:+.10f}')

# Feature importance
fig, ax = plt.subplots(figsize=(8, 4))
importance = np.abs(final_model.coef_[0])
sorted_idx = np.argsort(importance)
ax.barh([FEATURE_NAMES[i].replace('_', ' ') for i in sorted_idx], importance[sorted_idx], color='#2d5a3d')
ax.set_xlabel('|Weight| (absolute coefficient)')
ax.set_title('Feature Importance — Romberg Model')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Export model weights
precision_per_class = precision_score(y, y_pred_all, labels=['eyes_closed', 'eyes_open'], average=None)
recall_per_class = recall_score(y, y_pred_all, labels=['eyes_closed', 'eyes_open'], average=None)
f1_per_class = f1_score(y, y_pred_all, labels=['eyes_closed', 'eyes_open'], average=None)
cm_list = confusion_matrix(y, y_pred_all, labels=['eyes_closed', 'eyes_open']).tolist()

per_subj_export = [{'subject': r['subject'], 'accuracy': round(r['accuracy'], 4)}
                   for r in sorted(per_subject_results, key=lambda x: x['subject'])] if per_subject_results else []

romberg_export = {
    'model_type': 'logistic_regression',
    'task': 'romberg_eyes_open_closed',
    'feature_names': FEATURE_NAMES,
    'scaler_mean': final_scaler.mean_.tolist(),
    'scaler_std': final_scaler.scale_.tolist(),
    'weights': final_model.coef_[0].tolist(),
    'bias': float(final_model.intercept_[0]),
    'classes': list(final_model.classes_),
    'cv_accuracy': round(overall_acc, 4),
    'cv_type': cv_type,
    'cv_metrics': {
        'precision': {'eyes_closed': round(precision_per_class[0], 4), 'eyes_open': round(precision_per_class[1], 4)},
        'recall': {'eyes_closed': round(recall_per_class[0], 4), 'eyes_open': round(recall_per_class[1], 4)},
        'f1': {'eyes_closed': round(f1_per_class[0], 4), 'eyes_open': round(f1_per_class[1], 4)},
        'confusion_matrix': cm_list,
    },
    'per_subject_accuracy': per_subj_export,
    'dataset_info': {
        'n_subjects': n_subjects,
        'subjects': sorted(list(set(groups))),
        'n_windows': int(len(y)),
        'n_eyes_open': int(np.sum(y == 'eyes_open')),
        'n_eyes_closed': int(np.sum(y == 'eyes_closed')),
        'window_size_sec': WINDOW_SIZE / SAMPLE_RATE,
        'sample_rate_hz': SAMPLE_RATE,
        'gravity_removed': 'per-window mean subtraction',
    },
}

with open('romberg_model_weights.json', 'w') as f:
    json.dump(romberg_export, f, indent=2)

print('Model exported to romberg_model_weights.json')
print(f'\nExport summary:')
print(f'  Classes: {romberg_export["classes"]}')
print(f'  CV: {cv_type}, Accuracy: {romberg_export["cv_accuracy"]}')
print(f'  Subjects: {romberg_export["dataset_info"]["subjects"]}')
print(f'  Windows: {romberg_export["dataset_info"]["n_windows"]} ({romberg_export["dataset_info"]["n_eyes_open"]} open, {romberg_export["dataset_info"]["n_eyes_closed"]} closed)')

In [ ]:
# Cell 9: Verify exported model
with open('romberg_model_weights.json', 'r') as f:
    loaded = json.load(f)

w = np.array(loaded['weights'])
b = loaded['bias']
mu = np.array(loaded['scaler_mean'])
sigma = np.array(loaded['scaler_std'])
classes = loaded['classes']

def predict_from_json(feature_dict):
    x = np.array([feature_dict[f] for f in loaded['feature_names']])
    x_scaled = (x - mu) / sigma
    z = np.dot(w, x_scaled) + b
    prob = 1 / (1 + np.exp(-z))
    label = classes[1] if prob > 0.5 else classes[0]
    return label, prob

np.random.seed(42)
test_indices = np.random.choice(len(feature_df), min(20, len(feature_df)), replace=False)
mismatches = 0

for idx in test_indices:
    row = feature_df.iloc[idx]
    feat_dict = {f: row[f] for f in FEATURE_NAMES}
    json_label, json_prob = predict_from_json(feat_dict)
    
    x_sklearn = final_scaler.transform([X[idx]])
    sklearn_label = final_model.predict(x_sklearn)[0]
    
    if json_label != sklearn_label:
        mismatches += 1
        print(f'  MISMATCH at idx {idx}')

print(f'\nVerification: {len(test_indices) - mismatches}/{len(test_indices)} predictions match.')
if mismatches == 0:
    print('All predictions match — export is correct.')